<a href="https://colab.research.google.com/github/coderhouse2025-droid/Tablero-de-Control/blob/main/Tablero__de_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TABLERO DE CONTROL

Conexión a Google Drive (API) = Instalación (Bash)





In [1]:
pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib


Código para listar y descargar PDFs

In [12]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
import io, os
from google.colab import auth # Importar la librería de autenticación de Colab
import google.auth # AÑADIR ESTA LÍNEA
import google.auth.transport.requests # AÑADIR ESTA LÍNEA para refrescar tokens

def conectar_drive():
    creds = None
    SCOPES = ["https://www.googleapis.com/auth/drive"]

    # Intentar cargar las credenciales desde token.json
    if os.path.exists("token.json"):
        try:
            creds = Credentials.from_authorized_user_file("token.json", SCOPES)
            if not creds.valid:
                if creds.expired and creds.refresh_token:
                    # Refresh token if expired and refresh token exists
                    creds.refresh(google.auth.transport.requests.Request())
                else:
                    # If not valid and no refresh token, force re-authentication
                    creds = None
        except Exception as e:
            print(f"Error cargando token.json: {e}. Se intentará re-autenticar.")
            creds = None

    if not creds:
        # Si token.json no existe o las credenciales no son válidas, autenticar al usuario
        print("Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.")
        auth.authenticate_user()

        # Después de la autenticación de Colab, las credenciales se establecen en el entorno.
        # google.auth.default() las recuperará automáticamente.
        creds, project = google.auth.default(scopes=SCOPES)

        # Guardar las nuevas credenciales en token.json
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("drive", "v3", credentials=creds)

def descargar_pdfs(carpeta_id, destino="facturas/"):
    os.makedirs(destino, exist_ok=True)
    drive = conectar_drive()

    query = f"'{carpeta_id}' in parents and mimeType='application/pdf'"
    results = drive.files().list(q=query).execute()
    archivos = results.get("files", [])

    print(f"Encontrados {len(archivos)} PDFs en la carpeta de Drive.")
    if not archivos:
        print("No se encontraron PDFs en la carpeta especificada.")
        return []

    for f in archivos:
        print(f"Descargando: {f['name']}")
        request = drive.files().get_media(fileId=f["id"])
        fh = io.FileIO(os.path.join(destino, f["name"]), "wb")
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print("Descarga de PDFs completada.")
    return archivos

Lectura de PDFs (con OCR incluido)

Si los PDFs son nativos (texto), usamos pdfplumber.
Si son escaneados, usamos OCR con Tesseract.

In [3]:
pip install pdfplumber pytesseract pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.2 MB/s eta 0:00:00


Código híbrido (texto + OCR)

In [4]:
import pdfplumber
import pytesseract
from PIL import Image
import tempfile

def extraer_texto_pdf(path):
    texto = ""

    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                texto += page.extract_text() or ""
    except:
        pass

    if len(texto.strip()) < 20:  # si no hay texto → usar OCR
        import fitz  # PyMuPDF
        doc = fitz.open(path)
        for page in doc:
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            texto += pytesseract.image_to_string(img)

    return texto


Parsing y normalización de facturas

Para extraer:

Fecha

Monto

CUIT / proveedor

Categoría

Número de factura

In [5]:
import re
import pandas as pd

def parsear_factura(texto):
    fecha = re.search(r"\d{2}/\d{2}/\d{4}", texto)
    monto = re.search(r"\$ ?([\d\.,]+)", texto)
    proveedor = re.search(r"Proveedor: (.*)", texto)

    return {
        "fecha": fecha.group(0) if fecha else None,
        "monto": float(monto.group(1).replace(".", "").replace(",", ".")) if monto else None,
        "proveedor": proveedor.group(1).strip() if proveedor else None,
        "texto_completo": texto
    }


Dataset final

In [6]:
import glob

registros = []

for pdf in glob.glob("facturas/*.pdf"):
    texto = extraer_texto_pdf(pdf)
    data = parsear_factura(texto)
    data["archivo"] = pdf
    registros.append(data)

df = pd.DataFrame(registros)
df.to_csv("dataset_facturas.csv", index=False)


Insights y estadísticas automáticas

In [25]:
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
df["mes"] = df["fecha"].dt.to_period("M")

gasto_por_mes = df.groupby("mes")["monto"].sum()
mes_mayor_gasto = gasto_por_mes.idxmax() if not gasto_por_mes.empty and not gasto_por_mes.isnull().all() else None

gasto_por_proveedor = df.groupby("proveedor")["monto"].sum()
proveedor_top = gasto_por_proveedor.idxmax() if not gasto_por_proveedor.empty and not gasto_por_proveedor.isnull().all() else None

insights = {
    "gasto_total": df["monto"].sum(),
    "gasto_promedio": df["monto"].mean(),
    "mes_mayor_gasto": mes_mayor_gasto,
    "proveedor_top": proveedor_top
}


In [21]:
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google.colab import auth
import google.auth
import google.auth.transport.requests
import os

def conectar_drive():
    creds = None
    SCOPES = ["https://www.googleapis.com/auth/drive"]

    if os.path.exists("token.json"):
        try:
            creds = Credentials.from_authorized_user_file("token.json", SCOPES)
            if not creds.valid:
                if creds.expired and creds.refresh_token:
                    creds.refresh(google.auth.transport.requests.Request())
                else:
                    creds = None
        except Exception as e:
            print(f"Error cargando token.json: {e}. Se intentará re-autenticar.")
            creds = None

    if not creds:
        print("Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.")
        auth.authenticate_user()
        creds, project = google.auth.default(scopes=SCOPES)

    return build("drive", "v3", credentials=creds)

def listar_archivos_en_carpeta(folder_id):
    drive_service = conectar_drive()
    query = f"'{folder_id}' in parents"
    results = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    if not items:
        print(f"No se encontraron archivos en la carpeta con ID: {folder_id}")
    else:
        print(f"Archivos encontrados en la carpeta con ID: {folder_id}")
        for item in items:
            print(f"  - {item['name']} (ID: {item['id']}, Tipo: {item['mimeType']})")

# Usa la misma folder_id que usaste anteriormente
folder_id_a_verificar = '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo'
listar_archivos_en_carpeta(folder_id_a_verificar)


Error cargando token.json: Expecting value: line 1 column 1 (char 0). Se intentará re-autenticar.
Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.


Archivos encontrados en la carpeta con ID: 1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo
  - Edenor 0.pdf (ID: 1x7WegSUhRhPPnkTkKc5dZz_4jEQt1ah6, Tipo: application/pdf)
  - Edenor 1.pdf (ID: 1iI0B0KzOA8lsR6gLVkhwHd14R143t5EK, Tipo: application/pdf)
  - Personal 1.pdf (ID: 1vOt0nulTntOtCTWZi24nwPPu23geIUzM, Tipo: application/pdf)


In [27]:
import re
import pandas as pd
import glob
import io, os
import pdfplumber
import pytesseract
from PIL import Image
import tempfile

# Google Drive API imports
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google.colab import auth
import google.auth
import google.auth.transport.requests

# --- Conexión a Google Drive ---
def conectar_drive():
    SCOPES = ["https://www.googleapis.com/auth/drive"]
    print("Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.")
    auth.authenticate_user()
    creds, project = google.auth.default(scopes=SCOPES)
    return build("drive", "v3", credentials=creds)

# --- Descarga de PDFs ---
def descargar_pdfs(carpeta_id, destino="facturas/"):
    os.makedirs(destino, exist_ok=True)
    drive = conectar_drive()

    query = f"'{carpeta_id}' in parents and mimeType='application/pdf'"
    results = drive.files().list(q=query).execute()
    archivos = results.get("files", [])

    print(f"Encontrados {len(archivos)} PDFs en la carpeta de Drive.")
    if not archivos:
        print("No se encontraron PDFs en la carpeta especificada.")
        return []

    for f in archivos:
        print(f"Descargando: {f['name']}")
        request = drive.files().get_media(fileId=f["id"])
        fh = io.FileIO(os.path.join(destino, f["name"]), "wb")
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print("Descarga de PDFs completada.")
    return archivos

# --- Extracción de texto de PDFs (con OCR) ---
def extraer_texto_pdf(path):
    texto = ""

    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                texto += page.extract_text() or ""
    except:
        pass

    if len(texto.strip()) < 20:  # si no hay texto → usar OCR
        import fitz  # PyMuPDF
        doc = fitz.open(path)
        for page in doc:
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            texto += pytesseract.image_to_string(img)

    return texto

# --- Parseo de facturas ---
def parsear_factura(texto):
    fecha = re.search(r"\d{2}/\d{2}/\d{4}", texto)
    monto = re.search(r"\$ ?([\d\.,]+)", texto)

    proveedor = None
    # Try the explicit 'Proveedor:' label
    match_proveedor = re.search(r"Proveedor: (.*)", texto)
    if match_proveedor:
        proveedor = match_proveedor.group(1).strip()
    else:
        # Try to find specific company names in the text (case-insensitive)
        if re.search(r"Empresa Distribuidora y Comercializadora Norte|Edenor", texto, re.IGNORECASE):
            proveedor = "Edenor"
        elif re.search(r"Telecom|Personal|Fibertel", texto, re.IGNORECASE):
            proveedor = "Personal/Telecom" # Grouping similar services or explicit names
        # Add more patterns here for other common providers if needed

    return {
        "fecha": fecha.group(0) if fecha else None,
        "monto": float(monto.group(1).replace(".", "").replace(",", ".")) if monto else None,
        "proveedor": proveedor,
        "texto_completo": texto
    }

# --- Ejecución principal ---
folder_id = '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo'
descargar_pdfs(folder_id, destino="facturas/")
print(f"PDFs de la carpeta '{folder_id}' han sido descargados en la carpeta 'facturas/' en Colab.")

registros = []

for pdf in glob.glob("facturas/*.pdf"):
    texto = extraer_texto_pdf(pdf)
    data = parsear_factura(texto)
    data["archivo"] = pdf
    registros.append(data)

df = pd.DataFrame(registros)
df.to_csv("dataset_facturas.csv", index=False)

display(df.head())

# --- Insights y estadísticas automáticas ---
if not df.empty and 'fecha' in df.columns and 'monto' in df.columns:
    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
    df["mes"] = df["fecha"].dt.to_period("M")

    gasto_por_mes = df.groupby("mes")["monto"].sum()
    mes_mayor_gasto = gasto_por_mes.idxmax() if not gasto_por_mes.empty and not gasto_por_mes.isnull().all() else None

    gasto_por_proveedor = df.groupby("proveedor")["monto"].sum()
    proveedor_top = gasto_por_proveedor.idxmax() if not gasto_por_proveedor.empty and not gasto_por_proveedor.isnull().all() else None

    insights = {
        "gasto_total": df["monto"].sum(),
        "gasto_promedio": df["monto"].mean(),
        "mes_mayor_gasto": mes_mayor_gasto,
        "proveedor_top": proveedor_top
    }
    print("\nInsights generados:")
    for key, value in insights.items():
        print(f"- {key.replace('_', ' ').capitalize()}: {value}")
else:
    print("No se pudo generar insights: el DataFrame está vacío o faltan columnas clave (fecha, monto).")


Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.


Encontrados 3 PDFs en la carpeta de Drive.
Descargando: Edenor 0.pdf
Descargando: Edenor 1.pdf
Descargando: Personal 1.pdf
Descarga de PDFs completada.
PDFs de la carpeta '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo' han sido descargados en la carpeta 'facturas/' en Colab.


,fecha,monto,proveedor,texto_completo,archivo
0,26/01/2026,18585.13,Edenor,Empresa Distribuidora y Comercializadora Norte...,facturas/Edenor 0.pdf
1,24/02/2026,13513.42,Edenor,Empresa Distribuidora y Comercializadora Norte...,facturas/Edenor 1.pdf
2,None,NaN,None,Scanned by TapScanner,facturas/Personal 1.pdf



Insights generados:
- Gasto total: 32098.550000000003
- Gasto promedio: 16049.275000000001
- Mes mayor gasto: 2026-01
- Proveedor top: Edenor


In [18]:
folder_id = '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo'
descargar_pdfs(folder_id, destino="facturas/")
print(f"PDFs de la carpeta '{folder_id}' han sido descargados en la carpeta 'facturas/' en Colab.")

Error cargando token.json: Expecting value: line 1 column 1 (char 0). Se intentará re-autenticar.
Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.


Encontrados 0 PDFs en la carpeta de Drive.
No se encontraron PDFs en la carpeta especificada.
PDFs de la carpeta '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo' han sido descargados en la carpeta 'facturas/' en Colab.


In [17]:
folder_id = '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo'
descargar_pdfs(folder_id, destino="facturas/")
print(f"PDFs de la carpeta '{folder_id}' han sido descargados en la carpeta 'facturas/' en Colab.")

Error cargando token.json: Expecting value: line 1 column 1 (char 0). Se intentará re-autenticar.
Autenticando con Google Drive... Por favor, sigue las instrucciones en la ventana emergente.


Encontrados 0 PDFs en la carpeta de Drive.
No se encontraron PDFs en la carpeta especificada.
PDFs de la carpeta '1L7bllP0T8eTInqVWFzeAMhWEhFU9HSeo' han sido descargados en la carpeta 'facturas/' en Colab.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Una vez montado, podrás acceder a tus archivos de Google Drive a través de la ruta `/content/drive/MyDrive/`. Por ejemplo, si tienes una carpeta llamada `MisDocumentos` en tu Drive, podrás acceder a ella en Colab como `/content/drive/MyDrive/MisDocumentos/`.